# 065 — Deep (Stacked) RNNs

Stacking lets each layer work on the previous one's output rather than on raw
input. **Depth is the cheap axis**: on a 32-dimensional input, a second
`SimpleRNN(5)` costs **55** parameters while widening the first from 5 to 10
units costs **240**. And the rule that makes it work is one boolean —
`return_sequences=True` on every layer but the last.

| Part | What we check |
|---|---|
| A | depth against width, in parameters |
| B | what `return_sequences` actually controls |
| C | gradient decay along depth, against decay along time |
| D | a full stacked model, and where its weight sits |

## Part A — Depth is cheaper than width

In [ ]:
import numpy as np

def params(f_in, h):
    return f_in * h + h * h + h

print(f"  SimpleRNN(5) from a 32-dim input      {params(32, 5):>6,}")
print(f"  + a second SimpleRNN(5) on top        {params(5, 5):>6,}   <- depth")
print(f"  widening 5 -> 10 instead              "
      f"{params(32, 10) - params(32, 5):>6,}   <- width")
print(f"  widening 5 -> 50                      "
      f"{params(32, 50) - params(32, 5):>6,}")
assert params(5, 5) == 55 and params(32, 10) - params(32, 5) == 240
print()
print("A stacked layer's input is the previous layer's narrow output (5), not")
print("the wide embedding (32). That is the whole reason depth is cheap here -")
print(f"{240 / 55:.1f}x cheaper for the same 5 extra units.")

## Part B — The one boolean

A recurrent layer can hand on its **last** state or **every** state. The next
recurrent layer needs every state; a `Dense` needs only the last.

In [ ]:
T, H = 5, 16
states = np.arange(T * H).reshape(T, H)          # pretend these are hidden states

last_only = states[-1]
every = states
print(f"  return_sequences=False -> {last_only.shape}   (batch, H)")
print(f"  return_sequences=True  -> {every.shape}   (batch, T, H)")
assert last_only.shape == (H,) and every.shape == (T, H)

print("\n  what each consumer needs:")
for consumer, needs in (("another recurrent layer", "every state"),
                        ("Dense / classifier", "the last state"),
                        ("per-token tagger", "every state")):
    print(f"    {consumer:<24} {needs}")
print("\n  So: True on every recurrent layer except the last. Forget it and the")
print("  second layer receives a single vector where it expected a sequence -")
print("  the most common mistake in this section.")

## Part C — Gradients have two axes now

In [ ]:
H, F = 20, 8

def stacked(steps, layers, rho=0.9, seed=0):
    """Gradient norms along the depth axis, at the last time step."""
    r = np.random.default_rng(seed)
    weights = []
    for layer in range(layers):
        Wh = r.normal(0, 1, (H, H))
        Wh *= rho / max(abs(np.linalg.eigvals(Wh)))
        weights.append((r.normal(0, 0.3, (F if layer == 0 else H, H)), Wh))

    X = r.normal(0, 1, (steps, F))
    activations, inputs = [], X
    for Wx, Wh in weights:
        h, row = np.zeros(H), []
        for t in range(steps):
            h = np.tanh(inputs[t] @ Wx + h @ Wh)
            row.append(h.copy())
        activations.append(row)
        inputs = np.array(row)

    Wo = r.normal(0, 0.3, H)
    dh, norms = Wo.copy(), [np.linalg.norm(Wo)]
    for layer in range(layers - 1, 0, -1):
        Wx_layer, _ = weights[layer]
        dh = Wx_layer @ (dh * (1 - np.array(activations[layer][-1]) ** 2))
        norms.append(np.linalg.norm(dh))
    return np.array(norms)

print(f"  {'layers':>7}{'reaching layer 1':>20}{'fraction of top':>18}")
fractions = {}
for layers in (1, 2, 3, 4, 6):
    runs = [stacked(30, layers, seed=s) for s in range(40)]
    bottom = float(np.median([run[-1] for run in runs]))
    top = float(np.median([run[0] for run in runs]))
    fractions[layers] = bottom / top
    print(f"  {layers:>7}{bottom:>20.3e}{bottom / top:>18.3f}")
assert round(fractions[2], 2) == 0.81 and round(fractions[6], 2) == 0.49
print()
print("Real, and modest: a factor of two at six layers, where fifty time steps")
print("cost eleven orders of magnitude. Deep stacks use LSTM and GRU because of")
print("the TIME axis. Depth is not the reason.")

## Part D — A full stacked model

In [ ]:
V, D = 10_000, 32
rows = [("Embedding(10000, 32)", V * D),
        ("SimpleRNN(5, return_sequences=True)", params(D, 5)),
        ("SimpleRNN(5)", params(5, 5)),
        ("Dense(1)", 5 + 1)]
total = sum(p for _, p in rows)
for name, p in rows:
    print(f"  {name:<38}{p:>9,}  ({p / total:>5.1%})")
print(f"  {'TOTAL':<38}{total:>9,}")
recurrent = rows[1][1] + rows[2][1]
print(f"\n  embedding {V * D / total:.1%}, both recurrent layers together "
      f"{recurrent} parameters")
assert total == 320_251 and recurrent == 245
print()
print("The interesting layers are 245 parameters out of 320,251. Depth is not")
print("where the money goes - the vocabulary is.")

## What this notebook established

- A stacked `SimpleRNN(5)` costs **55** parameters; widening the first layer
  from 5 to 10 units costs **240**. Depth buys capacity about 4x more cheaply
  here, because a stacked layer's input is the previous layer's narrow output.
- `return_sequences=True` hands on `(batch, T, H)` instead of `(batch, H)` —
  required by another recurrent layer, wrong for a classifier head.
- Gradient reaching layer 1, median of 40 seeds: **0.81** of the top layer at 2
  layers, **0.49** at 6. Modest beside the time axis.
- In a full stacked model the embedding is **99.9%** of 320,251 parameters, and
  both recurrent layers together are **245**.

## Exercises

1. Build a 10-layer stack and measure the fraction reaching layer 1. Where does
   depth start to hurt?
2. Replace the second `SimpleRNN(5)` with `SimpleRNN(50)`. Recount, and say
   which axis you just bought.
3. Make the stack per-token (`return_sequences=True` everywhere). What does the
   output shape become, and what loss would you use?